# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 12: Análise de Ortogonalidade, Efeito Oráculo e Grau de Consenso na Classe MEDIUM

**Autor:** Evandro Rocha (UFRJ / COPPE / PEE)
**Data:** Outubro de 2026

---

## Resumo Executivo da Descoberta Científica

A classe **`MEDIUM`** (embarcações de porte intermediário: rebocadores oceânicos de porto, navios supridores de apoio à plataforma — *PSV/AHTS* e barcos de trabalho, comprimento entre $50\text{ m}$ e $100\text{ m}$) ocupa o centro geométrico do espaço de características acústicas do dataset IARA (*Silva et al.*, IEEE Access 2025).

Em classificadores isolados, o recall de `MEDIUM` atinge no máximo **$60{,}40\%$** (empatado entre **MEL** e **LOFAR**), enquanto o **DEMON** alcança $56{,}04\%$ e o **HHT Puro** atinge $44{,}03\%$.

Seguindo a investigação iniciada no Notebook 10 sobre `SMALL`, realizamos uma **análise exaustiva arquivo por arquivo** sobre todas as **1.490 instâncias de teste de `MEDIUM`** acumuladas nos 10 folds do protocolo oficial $5\times2$cv para o classificador **SVM Nyström Gaussiano**.

### Principais Descobertas:
1. **O Efeito Oráculo em MEDIUM ($83{,}69\%$ de Teto Teórico):**
   - Em **$1.247$ dos $1.490$ áudios de `MEDIUM` ($83{,}69\%$)**, pelo menos uma das representações acerta a classificação com sucesso!
   - Apenas **$16{,}31\%$ (243 áudios)** são erros comuns onde todas as 4 entradas falham simultaneamente (praticamente a mesma taxa de erro comum encontrada em `SMALL`, que foi de $16{,}01\%$).
2. **O Consenso em MEDIUM é 4× Maior que em SMALL ($27{,}05\%$ vs. $6{,}80\%$):**
   - Enquanto em `SMALL` apenas $6{,}8\%$ dos áudios eram acertados por todas as 4 entradas, em `MEDIUM` **$27{,}05\%$ (403 áudios)** são consensuais em todas as representações. O `MEDIUM` possui um "núcleo duro" acústico muito mais estável.
3. **O DEMON é o Campeão Absoluto de Resgates Exclusivos em MEDIUM:**
   - Em `SMALL`, o campeão de resgate exclusivo foi o **HHT** (207 lanchas salvas). Em `MEDIUM`, **o campeão é o DEMON**, salvando sozinho **107 áudios ($7{,}18\%$)** que MEL, LOFAR e HHT perdem simultaneamente!
   - *Razão Física:* Rebocadores operam hélices de passo variável com eixos lentos e fortes pás de propulsão ($400\text{ a }600\text{ RPM} \implies 6\text{ a }10\text{ Hz}$). O DEMON capta essa modulação de cavitação com nitidez cirúrgica.
4. **A Simetria Oculta da Confusão (O Canal `SMALL` $\leftrightarrow$ `MEDIUM`):**
   - Quando o MEL erra o `MEDIUM`, **$65{,}76\%$ dos erros vão para `SMALL`** (e $31{,}53\%$ para `LARGE`).
   - Como no Notebook 10 vimos que o MEL envia $49{,}26\%$ dos erros de `SMALL` para `MEDIUM`, existe um forte acoplamento acústico bidirecional entre lanchas em alta aceleração e rebocadores.
5. **A Votação por Maioria Simples ($\ge 2/4$) Funciona em MEDIUM ($64{,}63\%$):**
   - Diferente de `SMALL` (onde o ensemble simples falhava), em `MEDIUM` uma regra de maioria simples ($\ge 2$ votos) atinge **$64{,}63\%$ de recall**, superando o melhor modelo isolado ($60{,}40\%$) em **$+4{,}2$ pontos percentuais**.

In [ ]:
# 1. Configuração do Ambiente e Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

# Estilização publication-ready (sem dependência externa)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 150
})

print("Ambiente configurado com sucesso! Bibliotecas prontas.")

## 2. Metodologia Experimental e Base de Dados de Teste

Para assegurar paridade estrita com os experimentos anteriores:
- **Protocolo:** Validação cruzada $5\times2$cv (10 partições com isolamento estrito por `Ship ID`).
- **Classificador:** Support Vector Machine com Aproximação de Nyström Gaussiana (RBF, ElasticNet $\rho=0{,}15$, $m=4000/6000$ componentes).
- **Subconjunto de Teste:** Todas as $1.490$ instâncias onde a classe real comprovada era **`MEDIUM` (Classe 1)**.

Carregamos a base de predições consolidadas das 4 representações:

In [ ]:
# 2. Dados Consolidados das Predições do SVM na Classe MEDIUM
total_med = 1490

# Desempenho individual
hits_data = {
    'Representacao': ['HHT Puro (256-D)', 'DEMON (253-D)', 'LOFAR (1024-D)', 'MEL (256-D)', 'ORÁCULO (União 4 Modos)'],
    'Acertos': [656, 835, 900, 900, 1247],
    'Recall_Pct': [44.03, 56.04, 60.40, 60.40, 83.69]
}
df_hits = pd.DataFrame(hits_data)

# Grau de consenso
consensus_data = {
    'Nivel_Consenso': [
        '4 de 4 (Unanimidade / Independente)',
        '3 de 4 (Alto Consenso)',
        '2 de 4 (Consenso Parcial / Dividido)',
        '1 de 4 (Frágil / Resgate Exclusivo)',
        '0 de 4 (Erro Comum / Incorrigível)'
    ],
    'Audios': [403, 275, 285, 284, 243],
    'Pct_Total_Medium': [27.05, 18.46, 19.13, 19.06, 16.31],
    'Pct_Dentro_Oraculo': [32.32, 22.05, 22.85, 22.77, np.nan]
}
df_consensus = pd.DataFrame(consensus_data)

# Resgates exclusivos por representação
exclusive_data = {
    'Representacao': ['HHT Exclusivo', 'MEL Exclusivo', 'LOFAR Exclusivo', 'DEMON Exclusivo'],
    'Audios_Salvos': [43, 57, 77, 107],
    'Percentual': [2.89, 3.83, 5.17, 7.18]
}
df_exclusive = pd.DataFrame(exclusive_data)

# Efeito Reverso em Relação ao Baseline MEL (900 acertos, 590 erros)
reverse_data = {
    'Destino_Troca': ['Para LOFAR', 'Para DEMON', 'Para HHT'],
    'Mantem_do_MEL': [730, 641, 536],
    'Perde_do_MEL (Efeito Reverso)': [170, 259, 364],
    'Pct_Perda_Acertos (%)': [18.89, 28.78, 40.44],
    'Resgata_Erros_MEL': [170, 194, 120],
    'Pct_Resgate_Erros (%)': [28.81, 32.88, 20.34],
    'Saldo_Liquido_Audios': [0, -65, -244]
}
df_reverse = pd.DataFrame(reverse_data)

# Padrão de confusão quando erra
confusion_pattern_data = {
    'Representacao': ['MEL (590 erros)', 'DEMON (655 erros)', 'HHT (834 erros)', 'LOFAR (590 erros)'],
    'Confundido_SMALL_Pct': [65.76, 59.69, 63.91, 44.41],
    'Confundido_LARGE_Pct': [31.53, 27.33, 25.06, 43.56],
    'Confundido_AMBIENTAL_Pct': [2.71, 12.98, 11.03, 12.03]
}
df_confusion_pattern = pd.DataFrame(confusion_pattern_data)

display(df_hits)
display(df_consensus)
display(df_reverse)

## 3. O Efeito Oráculo na Classe MEDIUM

Na análise individual, **MEL** e **LOFAR** empatam exatamente no topo com **$60{,}40\%$ de recall** (900 acertos cada). O **DEMON** alcança $56{,}04\%$ e o **HHT Puro** fica com $44{,}03\%$.

Entretanto, ao calcularmos o **Teto Teórico do Oráculo** (união dos acertos onde pelo menos uma representação acerta o áudio):
- **O Recall de `MEDIUM` salta para $83{,}69\%$ (1.247 acertos em 1.490 áudios)!**
- Um ganho potencial de **$+23{,}3$ pontos percentuais** sobre o melhor modelo individual!
- Apenas **$16{,}31\%$ (243 áudios)** constituem o erro comum irrecuperável.

In [ ]:
# 3. Gráfico: Desempenho Isolado vs Teto Teórico do Oráculo em MEDIUM
plt.figure(figsize=(10, 5.5))
colors = ['#9b59b6', '#95a5a6', '#e67e22', '#2980b9', '#27ae60']
bars = plt.bar(df_hits['Representacao'], df_hits['Recall_Pct'], color=colors, width=0.55, edgecolor='black', linewidth=0.8)
bars[-1].set_hatch('//')

plt.axhline(83.69, color='#27ae60', linestyle='--', lw=1.5, alpha=0.7, label='Teto do Oráculo (83.69%)')
plt.axhline(60.40, color='#2980b9', linestyle=':', lw=1.5, alpha=0.7, label='Melhor Isolado MEL/LOFAR (60.40%)')

plt.title('Recall na Classe MEDIUM: Modelos Isolados vs. Teto Teórico do Oráculo', fontweight='bold', pad=12)
plt.ylabel('Recall em MEDIUM (%)')
plt.ylim(0, 100)

for bar in bars:
    h = bar.get_height()
    plt.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=10.5, fontweight='bold', color='#111')

plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 4. Análise de Grau de Consenso e Risco do Efeito Reverso em MEDIUM

### 4.1 A Pirâmide de Consenso em `MEDIUM` vs `SMALL`
A comparação entre as duas classes revela uma diferença estrutural marcante:
- **Unanimidade (4/4 certas):** Em `MEDIUM`, atinge **$27{,}05\%$ (403 áudios)**, enquanto em `SMALL` era de apenas **$6{,}80\%$**! Ou seja, o `MEDIUM` tem 4 vezes mais áudios acústica e fisicamente evidentes.
- **Alto Consenso (3/4 certas):** $275$ áudios ($18{,}46\%$).
- **Consenso Dividido (2/4 certas):** $285$ áudios ($19{,}13\%$).
- **Resgate Exclusivo (1/4 certa):** $284$ áudios ($19{,}06\%$) — em `SMALL` essa fatia era de quase $29\%$.
- **Erro Comum (0/4 certas):** $243$ áudios ($16{,}31\%$).

### 4.2 O Efeito Reverso a partir do MEL (900 Acertos de Referência)
Se um roteador desviar cegamente áudios do MEL para outra entrada:
1. **Chavear de MEL para LOFAR:**
   - Mantém **$730$ acertos ($81{,}11\%$)**.
   - Perde $170$ acertos ($18{,}89\%$) e resgata exatamente $170$ erros ($28{,}81\%$).
   - **Saldo Líquido: Zero ($+0$ áudios)!** Existe uma simetria de complementaridade balanceada entre MEL e LOFAR.
2. **Chavear de MEL para DEMON:**
   - Mantém **$641$ acertos ($71{,}22\%$)**.
   - Perde $259$ acertos e resgata $194$ erros. Saldo: $-65$ áudios.
3. **Chavear de MEL para HHT:**
   - Perde $364$ acertos e só resgata $120$. Saldo: $-244$ áudios.

### 4.3 O Sucesso da Votação por Maioria em MEDIUM
Diferente da classe `SMALL` (onde o ensemble simples afundava o recall), em `MEDIUM` a votação tradicional é muito mais eficaz:
- **Unanimidade (4/4):** $27{,}05\%$ ($403$ áudios).
- **Maioria Estrita ($\ge 3/4$):** $45{,}50\%$ ($678$ áudios).
- **Maioria Simples / Empate ($\ge 2/4$):** **$64{,}63\%$ ($963$ áudios)** $\rightarrow$ **Supera MEL e LOFAR isolados em $+4{,}2$ pontos percentuais!**
- **Oráculo ($\ge 1/4$):** **$83{,}69\%$ ($1.247$ áudios)**.

In [ ]:
# 4. Visualização do Grau de Consenso e do Efeito Reverso em MEDIUM
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Painel 1: Pirâmide de Consenso dos 1.490 Áudios
labels_cons = ['4 Entradas\n(Unânime)', '3 Entradas\n(Alto Consenso)', '2 Entradas\n(Dividido)', '1 Entrada\n(Exclusivo/Frágil)', '0 Entradas\n(Erro Comum)']
counts_cons = [403, 275, 285, 284, 243]
pcts_cons = [27.05, 18.46, 19.13, 19.06, 16.31]
colors_cons = ['#27ae60', '#2980b9', '#f39c12', '#e74c3c', '#7f8c8d']

bars1 = ax1.bar(labels_cons, counts_cons, color=colors_cons, edgecolor='black', linewidth=0.8, width=0.55)
ax1.set_title('Grau de Consenso em MEDIUM (Total: 1.490 Áudios no 5x2cv)', fontweight='bold', pad=12)
ax1.set_ylabel('Quantidade de Áudios de MEDIUM')
ax1.set_ylim(0, 480)

for bar, pct in zip(bars1, pcts_cons):
    h = bar.get_height()
    ax1.annotate(f'{h} ({pct:.1f}%)', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=9.5, fontweight='bold')

# Painel 2: Simulação do Efeito Reverso a partir do MEL (900 Acertos)
destinos = ['Para LOFAR', 'Para DEMON', 'Para HHT']
perdas = [170, 259, 364]
resgates = [170, 194, 120]
x = np.arange(len(destinos))
w = 0.35

r1 = ax2.bar(x - w/2, perdas, w, label='Perde Acertos do MEL (Efeito Reverso)', color='#c0392b', alpha=0.9, edgecolor='black')
r2 = ax2.bar(x + w/2, resgates, w, label='Resgata dos Erros do MEL (Ganhos)', color='#27ae60', alpha=0.9, edgecolor='black')

ax2.set_title('Balanço do Efeito Reverso ao Trocar de Entrada a partir do MEL', fontweight='bold', pad=12)
ax2.set_ylabel('Quantidade de Áudios de MEDIUM')
ax2.set_xticks(x)
ax2.set_xticklabels(destinos, fontweight='bold', fontsize=11)
ax2.set_ylim(0, 420)
ax2.legend(loc='upper right', frameon=True)

for rect in r1:
    h = rect.get_height()
    ax2.annotate(f'-{h}', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#900')

for rect in r2:
    h = rect.get_height()
    ax2.annotate(f'+{h}', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#006400')

plt.tight_layout()
plt.show()

## 5. Onde Mora o Valor: O Papel Central do DEMON nos Resgates Exclusivos

Em **$284$ áudios de `MEDIUM` ($19{,}06\%$)**, três entradas falharam simultaneamente e apenas uma representação conseguiu classificar o áudio corretamente.

A distribuição desses resgates exclusivos traz uma revelação fundamental:
1. **DEMON Isolado (107 rebocadores salvos exclusivamente — $7{,}18\%$):**
   - É o **líder incontestável de resgates em `MEDIUM`**!
   - *Diagnóstico Físico:* Rebocadores operando em manobra de porto ou apoio offshore possuem motores a diesel de rotação constante com hélice de passo controlável (*CPP*). A envoltória de cavitação DEMON ($0	ext{--}25	ext{ Hz}$) detecta as raias de passagem de pás com precisão cristalina, mesmo quando o ruído hidrodinâmico de casco confunde o espectro de banda larga.
2. **LOFAR Puro (77 rebocadores salvos exclusivamente — $5{,}17\%$):**
   - Embarcações `MEDIUM` possuem geradores auxiliares estacionários a diesel de $60\text{ Hz}$ e $120\text{ Hz}$. A alta resolução da FFT do LOFAR (1024 bins estreitos) captura essas raias elétricas puras.
3. **MEL Isolado (57 rebocadores salvos — $3{,}83\%$):**
   - Integra a potência contínua das descargas de exaustão submarina dos motores de propulsão.
4. **HHT Puro (43 rebocadores salvos — $2{,}89\%$):**
   - Ao contrário de `SMALL` (onde o HHT dominou com 207 lanchas salvas), em `MEDIUM` o casco pesado de aço e o regime de deslocamento geram menos instabilidades transientes não-lineares, reduzindo o papel isolado da HHT.

In [ ]:
# 5. Gráfico: Resgates Exclusivos na Classe MEDIUM
plt.figure(figsize=(9, 4.5))
colors_excl = ['#9b59b6', '#2980b9', '#e67e22', '#16a085']
bars_excl = plt.barh(df_exclusive['Representacao'], df_exclusive['Audios_Salvos'], color=colors_excl, height=0.5, edgecolor='black', linewidth=0.8)

plt.title('Resgates Exclusivos na Classe MEDIUM (Total: 284 Áudios Únicos)', fontweight='bold', pad=12)
plt.xlabel('Número de Áudios Salvos Exclusivamente')
plt.xlim(0, 140)

for idx, (bar, pct) in enumerate(zip(bars_excl, df_exclusive['Percentual'])):
    w = bar.get_width()
    tag = "  <-- CAMPEÃO DE RESGATES EM MEDIUM" if idx == 3 else ""
    plt.annotate(f'{w} áudios ({pct:.1f}%){tag}', xy=(w, bar.get_y() + bar.get_height()/2),
                 xytext=(5, 0), textcoords="offset points", ha='left', va='center',
                 fontsize=9.5, fontweight='bold', color='#111' if idx != 3 else '#117864')

plt.tight_layout()
plt.show()

## 6. Padrões de Erro em MEDIUM: A Dominância da Confusão com SMALL

Quando cada representação comete um erro em um áudio de `MEDIUM`, para qual classe ela o envia?

| Entrada | Principal Erro | Porcentagem do Erro | Segundo Maior Erro | Diagnóstico Mecânico |
| :--- | :---: | :---: | :---: | :--- |
| **MEL** | **`SMALL`** | **$65{,}76\%$** | `LARGE` ($31{,}53\%$) | O espectro contínuo suavizado de um rebocador em baixa potência mimetiza o perfil energético de uma lancha veloz. |
| **DEMON** | **`SMALL`** | **$59{,}69\%$** | `LARGE` ($27{,}33\%$) | Hélices de passo variável em manobra criam transientes de cavitação com modulações rápidas semelhantes a hélices pequenas. |
| **HHT** | **`SMALL`** | **$63{,}91\%$** | `LARGE` ($25{,}06\%$) | A decomposição empírica aloca energia em modos rápidos similares a cascos leves. |
| **LOFAR** | **`SMALL`** | **$44{,}41\%$** | **`LARGE` ($43{,}56\%$)** | **O LOFAR é o único que equilibra perfeitamente o erro:** suas raias estreitas identificam o porte intermediário dividindo o erro quase igualmente entre SMALL e LARGE. |

### A Descoberta do Canal Bidirecional `SMALL` $\leftrightarrow$ `MEDIUM`:
- No Notebook 10: o MEL confundia $49{,}26\%$ dos erros de `SMALL` como `MEDIUM`.
- Neste Notebook 12: o MEL confunde $65{,}76\%$ dos erros de `MEDIUM` como `SMALL`.
- **Conclusão:** Existe um eixo acústico fundamental no dataset IARA onde as lanchas e os rebocadores se mascaram reciprocamente sob transformadas baseadas em energia de Fourier (MEL e HHT). Apenas o **DEMON** e o **LOFAR** possuem a chave física para desatar esse nó!

In [ ]:
# 6. Gráfico: Destino do Erro em MEDIUM por Representação
plt.figure(figsize=(10, 5.5))
reps = df_confusion_pattern['Representacao']
sml_err = df_confusion_pattern['Confundido_SMALL_Pct']
lrg_err = df_confusion_pattern['Confundido_LARGE_Pct']
amb_err = df_confusion_pattern['Confundido_AMBIENTAL_Pct']

w = 0.52
p1 = plt.bar(reps, sml_err, w, label='Confundido com SMALL (Lanchas)', color='#3498db', alpha=0.9, edgecolor='black', linewidth=0.6)
p2 = plt.bar(reps, lrg_err, w, bottom=sml_err, label='Confundido com LARGE (Navios Grandes)', color='#e74c3c', alpha=0.9, edgecolor='black', linewidth=0.6)
p3 = plt.bar(reps, amb_err, w, bottom=np.array(sml_err)+np.array(lrg_err), label='Confundido com AMBIENTAL (Ruído)', color='#95a5a6', alpha=0.9, edgecolor='black', linewidth=0.6)

plt.title('Padrão de Erro na Classe MEDIUM: Para onde cada entrada envia o rebocador? (%)', fontweight='bold', pad=12)
plt.ylabel('Distribuição Percentual dos Erros (%)')
plt.ylim(0, 118)
plt.legend(loc='upper right', frameon=True)

for i in range(len(reps)):
    plt.text(i, sml_err[i]/2, f'{sml_err[i]:.1f}%', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    plt.text(i, sml_err[i] + lrg_err[i]/2, f'{lrg_err[i]:.1f}%', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if amb_err[i] > 4:
        plt.text(i, sml_err[i] + lrg_err[i] + amb_err[i]/2, f'{amb_err[i]:.1f}%', ha='center', va='center', fontsize=8.5, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 7. Painel Consolidado de Descoberta Científica

Abaixo exibimos o painel visual consolidado de 4 quadrantes que sintetiza a dinâmica de complementaridade da classe `MEDIUM`:

In [ ]:
# 7. Exibição do Gráfico Oficial Consolidado salvo em disco
from IPython.display import Image, display

img_path = 'reports/analise_complementaridade_medium_oraculo.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
else:
    print(f"Imagem {img_path} não encontrada. Execute 'scratch/plot_medium_complementarity.py' para gerá-la.")

## 8. Síntese Comparativa Cruzada: SMALL (Notebook 10) vs. MEDIUM (Notebook 12)

Ao confrontarmos lado a lado as descobertas obtidas nas duas classes críticas do dataset IARA, o panorama completo da acústica submarina de navios de menor tonelagem se torna evidente:

| Métrica / Dimensão Analítica | Classe `SMALL` (Notebook 10) | Classe `MEDIUM` (Notebook 12) | Implicação Arquitetural |
| :--- | :---: | :---: | :--- |
| **Total de Áudios Avaliados ($5\times2$cv)** | $1.780$ áudios | $1.490$ áudios | Cobertura integral das duas classes mais difíceis do IARA. |
| **Melhor Recall Individual Isolado** | **$50{,}73\%$** (MEL) | **$60{,}40\%$** (MEL e LOFAR) | `MEDIUM` tem maior recall base isolado que `SMALL`. |
| **Teto Teórico do Oráculo (União)** | **$83{,}99\%$** ($1.495$ áudios) | **$83{,}69\%$** ($1.247$ áudios) | **Convergência impressionante:** ambas as classes têm teto idêntico de $\approx 84\%$! |
| **Taxa de Erro Comum (Incorrigível)** | **$16{,}01\%$** ($285$ áudios) | **$16{,}31\%$** ($243$ áudios) | Apenas $\approx 16\%$ dos áudios são ruído cego em ambas as classes. |
| **Taxa de Unanimidade (4/4 certas)** | **$6{,}80\%$** ($121$ áudios) | **$27{,}05\%$** ($403$ áudios) | **$4\times$ maior em `MEDIUM`**: lanchas são muito mais sensíveis à representação. |
| **Campeão de Resgates Exclusivos** | **HHT Puro** ($207$ lanchas, $11{,}63\%$) | **DEMON** ($107$ rebocadores, $7{,}18\%$) | **Especialização ortogonal perfeita:** HHT salva `SMALL` e DEMON salva `MEDIUM`! |
| **Principal Destino do Erro** | `MEDIUM` ($49{,}3\%$) e `LARGE` ($52{,}6\%$) | **`SMALL`** ($65{,}8\%$) | Existe uma confusão cruzada recíproca entre `SMALL` e `MEDIUM`. |
| **Desempenho da Maioria Simples ($\ge 2/4$)** | $55{,}17\%$ (ganho modesto de $+4{,}4$ p.p.) | **$64{,}63\%$ (ganho sólido de $+4{,}2$ p.p.)** | Em `MEDIUM`, a votação simples já supera qualquer modelo isolado. |

---

## 9. Diretrizes Finais para o Roteador Global do Projeto IARA

As evidências empíricas dos Notebooks 10 e 12 estabelecem uma **estratégia unificada de inteligência multimodal**:

1. **O Núcleo de Desempate (`HHT` vs. `DEMON`):**
   - Quando o classificador base (**MEL**) estiver na fronteira de dúvida entre `SMALL` e `MEDIUM`:
     - Se o **HHT** indicar alta energia em frequências instantâneas transitórias de alta rotação $\longrightarrow$ **Decisão = `SMALL`**.
     - Se o **DEMON** acusar linha estável de rotação de pás na faixa de $6\text{ a }10\text{ Hz}$ $\longrightarrow$ **Decisão = `MEDIUM`**.
2. **Proteção contra o Efeito Reverso:**
   - O MEL deve permanecer como âncora; nenhum áudio com margem segura no MEL deve ser chaveado.
3. **Potencial de Impacto Global:**
   - Romper o teto de $50\%$ em `SMALL` e $60\%$ em `MEDIUM`, levando ambas as classes para a faixa de $70\text{--}80\%$, é o caminho definitivo para **elevar o Índice SP global do IARA para além de $70\%$**.